# 0. Objective

Explore the OpenYLD API for TRACE bond trading data.

Goals:
- Confirm authentication works
- Fetch TRACE records for a known-good date
- Understand the data schema and any quality issues
- Test pagination via sequence offset
- Probe historical coverage limits

## 0.1 Setup

Credentials are loaded from a `.env` file in this directory.
Copy `.env.example` to `.env` and fill in your credentials.
The `.env` file is gitignored and will never be committed.

In [25]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # loads .env from current working directory

EMAIL = os.environ.get("OPENYLD_EMAIL")
PASSWORD = os.environ.get("OPENYLD_PASSWORD")

if not EMAIL or not PASSWORD:
    raise EnvironmentError("OPENYLD_EMAIL and OPENYLD_PASSWORD must be set in .env")

print(f"Loaded credentials for: {EMAIL}")

Loaded credentials for: morgan@exponential-tech.ai


## 0.2 Authenticate

POST credentials to get a Bearer token.
The token is reused across all subsequent requests.

In [26]:
BASE_URL = "https://api.openyld.com"

auth_response = requests.post(
    f"{BASE_URL}/auth/login",
    json={"email": EMAIL, "password": PASSWORD}
)

if auth_response.status_code != 200:
    raise RuntimeError(f"Authentication failed ({auth_response.status_code}): {auth_response.text}")

token = auth_response.json().get("token")
headers = {"Authorization": f"Bearer {token}"}
print("Authentication successful.")

RuntimeError: Authentication failed (502): <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.24.0 (Ubuntu)</center>
</body>
</html>


# 1. Fetch TRACE Records for a Known-Good Date

The endpoint is:
```
GET /reported/historical/trace/{date}/{sequence}
```
- `date`: ISO format `YYYY-MM-DD`
- `sequence`: offset within that date's records (0 = start)

In [17]:
def fetch_trace(date: str, sequence: int = 0) -> list:
    url = f"{BASE_URL}/reported/historical/trace/{date}/{sequence}"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"  Error {response.status_code}: {response.text}")
        return []
    data = response.json()
    if isinstance(data, list):
        return data

    # Check for other common list keys
    for key in ["results", "data", "items", "records", "trades", "trace", "prints"]:
        if key in data:
            return data[key]

    # If we get here, print the raw payload so we can debug the exact schema
    print(f"  Debug - Unrecognized JSON structure. Keys available: {list(data.keys())}")
    print(f"  Debug - Raw response: {data}")
    return []


# Use the date from the sample CSV we already have
records = fetch_trace("2025-12-01")
if not records:
    # Fallback test: Try fetching yesterday's date to check if it's an entitlement/window issue
    from datetime import datetime, timedelta
    recent_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    print(f"\nTrying a recent date ({recent_date}) to test data entitlements...")
    records = fetch_trace(recent_date)
print(f"Records returned: {len(records)}")
if records:
    print("First record:")
    print(records[0])

Records returned: 500
First record:
{'sequence': 3, 'as_at': '2025-12-01T07:59:39', 'isin': 'US30303M8Y18', 'cusip': '30303M8Y1', 'side': 'Sell', 'quantity': '5000.0', 'price': '99.140000', 'yield': '5.8050', 'parties': 'D-A', 'is_ats': False}


# 2. Schema and Data Quality

Load records into a DataFrame and inspect types, missingness, and sentinel values.

Known issues from the sample CSV:
- `yield` uses `-999999` as a sentinel for missing/invalid
- `quantity` is stored as string, not numeric

In [18]:
df = pd.DataFrame(records)
print(df.dtypes)
print()
df.head()

sequence    int64
as_at         str
isin          str
cusip         str
side          str
quantity      str
price         str
yield         str
parties       str
is_ats       bool
dtype: object



,sequence,as_at,isin,cusip,side,quantity,price,yield,parties,is_ats
0,3,2025-12-01T07:59:39,US30303M8Y18,30303M8Y1,Sell,5000.0,99.140000,5.8050,D-A,False
1,4,2025-12-01T06:23:38,N/A,G8T22MAF7,Sell,250.0,97.900000,5.4904,D-A,False
2,5,2025-11-28T15:30:06,US037833DK32,037833DK3,Sell,100.0,99.081000,3.4910,D-A,False
3,6,2025-11-28T15:30:06,US89236TKL88,89236TKL8,Sell,100.0,103.018000,3.8211,D-A,False
4,7,2025-11-28T15:30:05,US06406RBP10,06406RBP1,Sell,100.0,101.440000,4.4680,D-A,False


In [19]:
# Clean known issues
if not df.empty:
    if "yield" in df.columns:
        df["yield"] = pd.to_numeric(df["yield"], errors="coerce")
        df.loc[df["yield"] == -999999, "yield"] = float("nan")

    if "quantity" in df.columns:
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

    print("Missing values after cleaning:")
    print(df.isnull().sum())
    print()
    display(df.describe())
else:
    print("DataFrame is empty. Nothing to clean or describe.")

Missing values after cleaning:
sequence    0
as_at       0
isin        0
cusip       0
side        0
quantity    0
price       0
yield       0
parties     0
is_ats      0
dtype: int64



,sequence,quantity,yield
count,500.000000,500.000000,500.000000
mean,253.874000,418.258046,7.199029
std,145.496118,960.718548,13.928541
min,3.000000,2.000000,-1.400000
25%,127.750000,20.000000,4.193825
50%,253.500000,100.000000,4.948050
75%,379.250000,250.000000,5.534225
max,507.000000,5000.001000,107.928700


# 3. Pagination Test

Each response may return a page of records. Loop through pages using
the `provider_sequence` of the last record as the next offset,
until an empty page is returned.

In [20]:
def fetch_all_trace(date: str) -> pd.DataFrame:
    all_records = []
    sequence = 0
    page = 0

    while True:
        batch = fetch_trace(date, sequence)
        if not batch:
            break
        all_records.extend(batch)
        page += 1
        print(f"  Page {page}: {len(batch)} records (total so far: {len(all_records)})")

        # Advance sequence cursor to one past the last record's sequence
        last_record = batch[-1]
        last_seq = last_record.get("sequence", last_record.get("sequenceNumber", last_record.get("provider_sequence")))
        if last_seq is None or last_seq <= sequence:
            print(f"  Stopping: Could not find valid next sequence (last_seq={last_seq})")
            break
        sequence = last_seq + 1

    df_all = pd.DataFrame(all_records)
    return df_all


print("Fetching all TRACE records for 2025-12-01...")
df_full = fetch_all_trace("2025-12-01")
print(f"\nTotal records: {len(df_full)}")
df_full.head()

Fetching all TRACE records for 2025-12-01...
  Page 1: 500 records (total so far: 500)
  Page 2: 500 records (total so far: 1000)
  Page 3: 500 records (total so far: 1500)
  Page 4: 500 records (total so far: 2000)
  Page 5: 500 records (total so far: 2500)
  Page 6: 500 records (total so far: 3000)
  Page 7: 500 records (total so far: 3500)
  Page 8: 500 records (total so far: 4000)
  Page 9: 500 records (total so far: 4500)
  Page 10: 500 records (total so far: 5000)
  Page 11: 500 records (total so far: 5500)
  Page 12: 500 records (total so far: 6000)
  Page 13: 500 records (total so far: 6500)
  Page 14: 500 records (total so far: 7000)
  Page 15: 500 records (total so far: 7500)
  Page 16: 500 records (total so far: 8000)
  Page 17: 500 records (total so far: 8500)
  Page 18: 500 records (total so far: 9000)
  Page 19: 500 records (total so far: 9500)
  Page 20: 500 records (total so far: 10000)
  Page 21: 500 records (total so far: 10500)
  Page 22: 500 records (total so far: 1

KeyboardInterrupt: 

# 4. Historical Coverage Probe

Test a range of dates to find the earliest date with data.
We already know 2012 returns nothing — this confirms the coverage start.

In [22]:
probe_dates = [
    "2012-01-03",
    "2020-01-02",
    "2021-01-04",
    "2022-01-03",
    "2023-01-03",
    "2024-01-02",
    "2025-01-02",
    "2025-12-01",
]

coverage = []
for date in probe_dates:
    records = fetch_trace(date)
    coverage.append({"date": date, "record_count": len(records)})
    print(f"  {date}: {len(records)} records")

pd.DataFrame(coverage)

  2012-01-03: 0 records
  2020-01-02: 0 records
  2021-01-04: 0 records
  2022-01-03: 0 records
  2023-01-03: 500 records
  2024-01-02: 500 records
  2025-01-02: 500 records
  2025-12-01: 500 records


,date,record_count
0,2012-01-03,0
1,2020-01-02,0
2,2021-01-04,0
3,2022-01-03,0
4,2023-01-03,500
5,2024-01-02,500
6,2025-01-02,500
7,2025-12-01,500


# 5. Summary

Document findings:
- Coverage start date
- Schema fields
- Data quality issues and required cleaning steps
- Pagination behaviour

In [ ]:
print("Schema fields:", list(df_full.columns) if not df_full.empty else "N/A")
print("Coverage start: see probe table above")
print("Known cleaning required:")
print("  - yield: -999999 sentinel → NaN")
print("  - quantity: string → numeric")